> **Multi-node workspace required:** To run this example, your workspace must have
> have multi-node enabled. Contact [support@modal.com](mailto:support@modal.com) to get access.

# Multi-node Kimi K2.5 LoRA training

This tutorial runs LoRA GRPO on
[Kimi-K2.5](https://huggingface.co/moonshotai/Kimi-K2.5), a
Mixture-of-Experts model from Moonshot AI, using
[miles](https://github.com/volcengine/verl) across **16 nodes
(128 H200 GPUs)**.

The `Kimi_K2_5_LoRA_Recipe` preset configures INT4-quantized
training weights with BF16 reference weights, colocated actor/critic,
and DeepScaler math reward verification.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import modal

from modal_training_gym.common.modal_urls import modal_app_dashboard_url
from modal_training_gym import (
    HuggingFaceDataset,
    Kimi_K2_5,
    Kimi_K2_5_LoRA_Recipe,
    TrainConfig,
)

## Dataset

We use [DAPO-Math-17k](https://huggingface.co/datasets/zhuzilin/dapo-math-17k),
a collection of math competition problems with verifiable answers.
The `deepscaler` reward model checks whether the model's response
matches the reference answer.

In [ ]:
class MathDataset(HuggingFaceDataset):
    hf_repo = "zhuzilin/dapo-math-17k"
    input_column = ""
    output_column = ""
    input_key = "prompt"
    label_key = "label"
    output_format = "jsonl"
    apply_chat_template = True
    always_prepare = True

## Build and launch training

Build the training config, construct the Modal app, and spawn
the training function as a detached call.

In [ ]:
def build_training_config() -> TrainConfig:
    return TrainConfig(
        model=Kimi_K2_5(),
        dataset=MathDataset(n_rows=10),
        recipe=Kimi_K2_5_LoRA_Recipe(),
    )

training_run = build_training_config()
app = training_run._build_app()

with modal.enable_output():
    with app.run():
        modal_app_id = app.app_id or ""
        function_call = app.train.spawn(
            modal_app_id=modal_app_id,
            modal_app_url=modal_app_dashboard_url(modal_app_id),
        )
        print(f"Spawned train function call: {function_call.object_id}")